In [19]:
from rdflib import Graph,URIRef,Namespace,BNode,Literal
from rdflib.namespace import XSD,RDF
import pandas as pd
import argparse
from collections import defaultdict
from datetime import datetime
import json

In [20]:
graph = Graph()
print("Started loading graph...")
graph.parse('sample.ttl', format="turtle",publicID="https://example.org/")
print("Graph loaded successfully.")

Started loading graph...
Graph loaded successfully.


In [47]:
get_sensor_query = ''' 
PREFIX sosa: <http://www.w3.org/ns/sosa/>

SELECT DISTINCT ?sensor
WHERE {
  ?s sosa:madeBySensor ?sensor .
}
'''
sensor_set = set()
sensor_set.clear()
for sensor in graph.query(get_sensor_query):
    print(sensor.sensor)
    sensor_set.add(str(sensor[0]))

https://example.org/temp_sensor_1
https://example.org/humidity_sensor_1
https://example.org/temp_sensor_2
https://example.org/pressure_sensor_1
https://example.org/light_sensor_1
https://example.org/wind_sensor_1
https://example.org/precipitation_sensor_1
https://example.org/humidity_sensor_2
https://example.org/pressure_sensor_2
https://example.org/light_sensor_2
https://example.org/wind_sensor_2


In [40]:
prefix_tss = Namespace('https://w3id.org/tss#')
prefix_ex  = Namespace('http://example.org/')

In [55]:
final_graph = Graph()
final_graph.bind('tss',prefix_tss)
final_graph.bind('ex',prefix_ex)


for sensor in sensor_set:
    tss_points = []
    test_query = f'''
    PREFIX sosa: <http://www.w3.org/ns/sosa/>

    SELECT ?READING ?TIME ?OBSERVATION  
    WHERE {{
        ?OBSERVATION a sosa:Observation ;
           sosa:resultTime ?TIME;
           sosa:hasSimpleResult ?READING;
           sosa:madeBySensor <{sensor}>.

    }}

    ORDER BY ?TIME
    '''
    results = graph.query(test_query)
    for row in results:
        #print(sensor, ' value:' ,row.READING, ' time:', row.TIME, ' ID:' ,row.OBSERVATION)

        
        data = {
            'time': row.TIME,
            'value': row.READING,
            'id': row.OBSERVATION
        }
        
        tss_points.append(data)
    json_object = json.dumps(tss_points) #serialize json object to a string
    
    #Create new graph 
    subject  = prefix_ex[f"snippet/{str(tss_points[0]['time'])[:9]}"] #this is the proper subject and should replace "URIRef(sensor)"
    final_graph.add((URIRef(sensor),RDF.type,prefix_tss.Snippet)) #temp
    final_graph.add((URIRef(sensor),prefix_tss.points,Literal(json_object, datatype=RDF.JSON)))
    final_graph.add((URIRef(sensor),prefix_tss["from"],tss_points[0]['time'])) #from is a reserved word, hence worked around it this way
    final_graph.add((URIRef(sensor),prefix_tss.to,tss_points[-1]['time'])) #from is a reserved word, hence worked around it this way


    '''
    print(json_object)
    print('start time: ', tss_points[0]['time']) #since readings are already sorted, first object holds start time
    print('end time: ', tss_points[-1]['time']) #while last object always holds last time
    print('-----------------------------------------------------------------------------------------------------------------------')
    '''

In [60]:
for subj, pred, obj in final_graph:
    print(subj, pred, obj)

https://example.org/pressure_sensor_2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://w3id.org/tss#Snippet
https://example.org/light_sensor_2 https://w3id.org/tss#from 2026-05-20T20:20:20+00:00
https://example.org/wind_sensor_2 https://w3id.org/tss#from 2026-05-31T23:59:59+00:00
https://example.org/wind_sensor_2 https://w3id.org/tss#to 2026-05-31T23:59:59+00:00
https://example.org/pressure_sensor_1 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://w3id.org/tss#Snippet
https://example.org/humidity_sensor_1 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://w3id.org/tss#Snippet
https://example.org/wind_sensor_2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://w3id.org/tss#Snippet
https://example.org/light_sensor_1 https://w3id.org/tss#to 2026-02-14T12:14:14+00:00
https://example.org/precipitation_sensor_1 https://w3id.org/tss#to 2026-03-15T05:05:05+00:00
https://example.org/temp_sensor_1 https://w3id.org/tss#to 2026-02-14T14:30:15+00:00
https://example.org/temp_